# Assignment 06 · Notebook 05: bốn mô hình bằng Keras trên cả hai tập

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Cùng kiến trúc (tầng hồi quy 64 → Dense 32 ReLU → Dense 1), cùng AdamW với `clipnorm=1.0`, cùng lô, cùng dừng sớm
(`EarlyStopping(restore_best_weights=True)`), cùng dữ liệu đã chuẩn hoá như PyTorch. Backend TensorFlow chạy trên
**CPU** (TensorFlow ≥ 2.11 không hỗ trợ GPU trên Windows), 16 luồng, song song với notebook 03–04 đang dùng GPU.

In [1]:
import sys
sys.path.insert(0, "..")
from a06.models_keras import configure_tf, run_keras
configure_tf(16)
import keras
import pandas as pd
import tensorflow as tf
from a06.bench import eval_churn, eval_stock, naive_stock, save_run
from a06.data_churn import load_churn
from a06.data_stock import prepare_stock
from a06.export import save_metrics
from a06.models_torch import CELLS, CELL_LABELS

print("TensorFlow", tf.__version__, "· Keras", keras.__version__, "· GPU thấy được:", tf.config.list_physical_devices("GPU"))

TensorFlow 2.21.0 · Keras 3.15.1 · GPU thấy được: []


## AMZN

In [2]:
S = prepare_stock()
out_s = run_keras("amzn", S)
meta_s = save_run("amzn", "keras", out_s)
res_s = {c: eval_stock(S, out_s[c]["pred_test"]) for c in CELLS}
pd.DataFrame({CELL_LABELS[c]: {k: res_s[c][k] for k in ("rmse", "mae", "r2", "r2_ret", "dir_acc")} for c in CELLS}).T.round(4)

amzn keras rnn    epoch tốt  2/10 val 0.2346 · 0.82 s/epoch trên cpu


amzn keras lstm   epoch tốt  1/9 val 0.2233 · 1.16 s/epoch trên cpu


amzn keras gru    epoch tốt  2/10 val 0.2239 · 1.29 s/epoch trên cpu


amzn keras bilstm epoch tốt  1/9 val 0.2304 · 2.08 s/epoch trên cpu


,rmse,mae,r2,r2_ret,dir_acc
Simple RNN,3.2174,2.3575,0.9874,-0.0477,0.4980
LSTM,3.1321,2.2817,0.9880,-0.0037,0.5117
GRU,3.1321,2.2754,0.9880,-0.0041,0.5215
BiLSTM,3.1401,2.2835,0.9880,-0.0136,0.5166


## KKBox Churn

In [3]:
C = load_churn()
out_c = run_keras("kkbox", C)
meta_c = save_run("kkbox", "keras", out_c)
res_c = {c: eval_churn(C, out_c[c]["pred_val"], out_c[c]["pred_test"]) for c in CELLS}
pd.DataFrame({CELL_LABELS[c]: {k: v for k, v in res_c[c].items() if k != "confusion"} for c in CELLS}).T.round(4)

kkbox keras rnn    epoch tốt  5/9 val 1.1576 · 2.60 s/epoch trên cpu


kkbox keras lstm   epoch tốt  7/11 val 1.1310 · 9.87 s/epoch trên cpu


kkbox keras gru    epoch tốt 15/19 val 1.1295 · 8.26 s/epoch trên cpu


kkbox keras bilstm epoch tốt  8/12 val 1.1322 · 17.55 s/epoch trên cpu


,acc,precision,recall,f1,roc_auc,pr_auc,threshold
Simple RNN,0.8991,0.4051,0.2701,0.3241,0.6616,0.2697,0.6489
LSTM,0.9022,0.4294,0.2783,0.3377,0.6787,0.3046,0.7108
GRU,0.8976,0.4026,0.2954,0.3408,0.6797,0.3091,0.6940
BiLSTM,0.8983,0.4068,0.2939,0.3413,0.6776,0.3028,0.6878


In [4]:
keep = ("params", "best_epoch", "epochs_run", "epoch_s", "train_s")
save_metrics("keras", {
    "amzn": {c: {**{k: v for k, v in res_s[c].items() if k != "price_pred"},
                 **{k: meta_s[f"amzn_keras_{c}"][k] for k in keep}} for c in CELLS},
    "kkbox": {c: {**res_c[c], **{k: meta_c[f"kkbox_keras_{c}"][k] for k in keep}} for c in CELLS}})
print("đã ghi outputs/metrics/keras.json")

đã ghi outputs/metrics/keras.json
